In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("../data/daangn_nintendo_full.csv")
len(df)


2793

In [ ]:
print(df.columns.tolist())

['gu', 'dong', 'title', 'price', 'link', 'status', 'is_OLED', 'is_Lite', 'is_Edition', 'price_clean', 'content', 'posted_time']


In [13]:
# title과 dong이 같은 항목 제거
df = df.drop_duplicates(subset=['title', 'dong'], keep='first')

In [15]:
# 10만원 아래의 항목을 전부 제거
df = df[df['price'] >= 100000]

In [17]:
# 버전에 따른 플래그 작업 실시(OLED, Lite, 동물의숲 에디션)
df['is_OLED']    = df['title'].str.contains('OLED|oled', case=False, na=False)
df['is_Lite']    = df['title'].str.contains('Lite|lite', case=False, na=False)
df['is_Edition'] = df['title'].str.contains('에디션|애디션', case=False, na=False)

In [19]:
df['price_clean'] = (df['price']
    .astype(str)
    .str.replace(r"[^\d]", "", regex=True)
    .pipe(pd.to_numeric, errors='coerce'))


In [20]:
df.to_csv("../data/daangn_nintendo_cleaned.csv", index=False)

In [5]:
df.drop(columns=["price"], inplace=True)

In [7]:
df.to_csv("../data/daangn_nintendo_cleaned.csv", index=False)

In [9]:
df[['title', 'content']].isnull().sum()

title       0
content    15
dtype: int64

In [10]:
df = df[~df["content"].isna()]

In [11]:
df[['title', 'content']].isnull().sum()

title      0
content    0
dtype: int64

In [12]:
df.to_csv("../data/daangn_nintendo_cleaned.csv", index=False)

In [13]:
df['price_clean'].describe()


count    2.778000e+03
mean     2.560268e+06
std      2.504514e+06
min      1.000000e+06
25%      1.900000e+06
50%      2.500000e+06
75%      3.000000e+06
max      1.234568e+08
Name: price_clean, dtype: float64

In [15]:
df[df["price_clean"] > 10_000_000].sort_values("price_clean", ascending=False)

,gu,dong,title,link,status,is_OLED,is_Lite,is_Edition,price_clean,content,posted_time
1490,분당구,야탑1동,닌텐도 스위치 게임 팝니다,https://www.daangn.com/kr/buy-sell/%EB%8B%8C%E...,판매완료,False,False,False,123456780,"닌텐도 스위치 중고 게임 팝니다 1, 베어너클4 : 25,000원 2, 노 모어 ...",2025-02-16
476,강남구,대치4동,[닌텐도] 스위치 구형 및 게임칩 33종 & 기타 물건 판매,https://www.daangn.com/kr/buy-sell/%EB%8B%8C%E...,판매완료,False,False,False,18000000,"구매일자 : 2018년 상태 : B+급 판매사유 : 회사 대표님, 아들이 커서 더 ...",2024-05-20
2768,부산진구,개금제2동,엔진11 크릿디 캔디블루 중급구성 M사이즈 판매만,https://www.daangn.com/kr/buy-sell/%EC%97%94%E...,판매완료,False,False,False,14000000,엔진11 크릿디 캔디블루 네고가능 타이어 끝에 야광같은것 같은데 알루휠 아닙니다 제...,2025-01-27
5,강남구,청담동,닌텐도 스위치 칩 미사용 제품들 팝니다!,https://www.daangn.com/kr/buy-sell/%EB%8B%8C%E...,판매중,False,False,False,11111110,전부 겉비닐만 까진 미사용한 보관만한 제품들입니다. 미사용 제품인데 ㅜㅜ오지게 쓴...,2025-03-23
2452,부산진구,전포제2동,닌텐도 스위치 칩,https://www.daangn.com/kr/buy-sell/%EB%8B%8C%E...,판매완료,False,False,False,11111110,1.마리오 종비접기 킹 3.5 2.슈퍼 스매시브라더스 얼티밋 4.0,2024-09-27
658,강남구,삼성1동,(급처)아이폰14프로 딥퍼플 128GB+닌텐도 스위치 신형,https://www.daangn.com/kr/buy-sell/%EA%B8%89%E...,판매중,False,False,False,11000000,"아이폰 14프로 딥퍼플 128GB (23.2월 구매) 모서리,액정 깨짐 없습니다 배...",2024-06-06


In [22]:
df = df[df["price_clean"] <= 1000000]

In [17]:
import re
def clean_text(text):
    text = re.sub(r'[^\w\s가-힣]', ' ', text)  # 한글, 숫자, 영문자 제외 제거
    text = re.sub(r'\s+', ' ', text).strip()  # 중복 공백 제거
    return text

df["title"] = df["title"].astype(str).apply(clean_text)
df["content"] = df["content"].astype(str).apply(clean_text)

In [19]:
df["status"].value_counts()

status
판매완료    2427
판매중      295
예약중       50
Name: count, dtype: int64

In [20]:
df["posted_time"] = pd.to_datetime(df["posted_time"])

In [25]:
def detect_set(row):
    text = (row['title'] + ' ' + row['content']).lower()
    keywords_set = ['세트', '풀셋', '풀박', '풀구성', '일괄', '같이', '+', '게임 포함']
    return any(keyword in text for keyword in keywords_set)

In [26]:
df['is_set'] = df.apply(detect_set, axis=1)

In [27]:
df.to_csv("../data/daangn_nintendo_cleaned.csv", index=False)

In [28]:
set_items_df = df[df["is_set"] == True]

In [29]:
set_items_df.head()

,gu,dong,title,link,status,is_OLED,is_Lite,is_Edition,price_clean,content,posted_time,is_set
459,강남구,대치4동,닌텐도 스위치 마리오카트 팝니다,https://www.daangn.com/kr/buy-sell/%EB%8B%8C%E...,판매완료,False,False,False,1000000,초기화한 닌텐도 스위치랑 마리오카트 게임 칩과 같이 10만원에 팔아요 본체랑 칩만 ...,2024-11-06,True
860,강남구,도곡1동,닌텐도 스위치 팝니다,https://www.daangn.com/kr/buy-sell/%EB%8B%8C%E...,판매완료,False,False,False,1000000,구형이여서 좀 더 싸게 팝니다 제조일자2018 9 작동에는 아무 이상 없습니다 스위...,2024-08-02,True
1136,분당구,판교동,닌텐도 스위치 링피트 패키지,https://www.daangn.com/kr/buy-sell/%EB%8B%8C%E...,판매완료,False,False,False,1000000,닌텐도 스위치 링피트 패키지 판매합니다 남편이 링핏 한다고 사놓은 풀 셋트 한달 사...,2025-04-06,True
1732,분당구,정자2동,마리오 래비드 닌텐도 스위치,https://www.daangn.com/kr/buy-sell/%EB%A7%88%E...,판매완료,False,False,False,1000000,일괄 100 000원 마리오 래비드 골드 시즌패스도 포함,2023-09-04,True
1909,부산진구,전포동,닌텐도 스위치 라이트,https://www.daangn.com/kr/buy-sell/%EB%8B%8C%E...,판매완료,False,False,False,1000000,잘 안하게되어 판매하게되었습니다 라이트 본체는 보호필름부착되어있고 사용감은 좀 있으...,2024-08-24,True
